# S5 Causal Masked Reconstruction Colab

This notebook trains a causal `S5` encoder with masked signal reconstruction on Utah-array cache shards stored in Google Drive.

Default design:

- reconstruct normalized raw patch values in `TX` space
- causal `S5` backbone with session-keyed token read-in / read-out boundaries
- patch-level masking by default with `patch_size=4`, `patch_stride=2`
- contiguous masked spans with a fixed overall mask ratio
- same held-out `Brain2Text25` frozen phoneme probe workflow used in the other `s5` notebooks


In [17]:
# Mount Drive and resolve cache / output roots.

from google.colab import drive
from pathlib import Path

drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive")
cache_candidates = [
    DRIVE_ROOT / "utah_ssl" / "data" / "cache_v1",
]

CACHE_ROOT = next((p for p in cache_candidates if p.exists()), cache_candidates[0])
OUTPUT_ROOT = DRIVE_ROOT / "utah_ssl" / "outputs" / "ssl_experiments" / "masked_reconstruction"
OUTPUT_ROOT.mkdir(parents=True, exist_ok=True)

print("DRIVE_ROOT :", DRIVE_ROOT)
print("CACHE_ROOT :", CACHE_ROOT, "| exists:", CACHE_ROOT.exists())
print("OUTPUT_ROOT:", OUTPUT_ROOT, "| exists:", OUTPUT_ROOT.exists())

if CACHE_ROOT.exists():
    datasets = sorted(p.name for p in CACHE_ROOT.iterdir() if p.is_dir())
    print("datasets:", datasets)
else:
    print("cache candidates checked:")
    for path in cache_candidates:
        print(" -", path)


Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).
DRIVE_ROOT : /content/drive/MyDrive
CACHE_ROOT : /content/drive/MyDrive/utah_ssl/data/cache_v1 | exists: True
OUTPUT_ROOT: /content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/masked_reconstruction | exists: True
datasets: ['000950', 'brain2text24', 'brain2text25', 'motor_data', 'plug_n_play', 'unsupervised_cursor_recalibration_offline', 'unsupervised_cursor_recalibration_online', 'willett_handwriting']


In [18]:
# Clone the public repo and import the reusable masked SSL helpers.

import os
import subprocess
import sys

REPO_URL = "https://github.com/ethan-read/utah-ssl.git"
REPO_DIR = Path("/content/utah-ssl")
EXPERIMENTS_DIR = REPO_DIR / "analysis" / "active" / "ssl_experiments"
MASKED_SSL_DIR = EXPERIMENTS_DIR / "masked_ssl"
SSL_DIR = REPO_DIR / "analysis" / "active" / "transfer_benchmark" / "ssl_autoresearch"

os.chdir("/content")

if REPO_DIR.exists():
    print("Using existing repo:", REPO_DIR)
else:
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)

os.chdir(REPO_DIR)

for candidate in (REPO_DIR, EXPERIMENTS_DIR, SSL_DIR):
    candidate_str = str(candidate)
    if candidate_str not in sys.path:
        sys.path.insert(0, candidate_str)

os.environ["SSL_AUTORESEARCH_CACHE_ROOT"] = str(CACHE_ROOT)
os.environ["SSL_AUTORESEARCH_OUTPUT_ROOT"] = str(OUTPUT_ROOT)

if not MASKED_SSL_DIR.exists():
    raise FileNotFoundError(
        "The cloned repo does not contain analysis/active/ssl_experiments/masked_ssl. "
        "Make sure REPO_DIR points at a checkout that includes the masked reconstruction package."
    )

from masked_ssl import (
    CacheAccessConfig,
    DownstreamProbeConfig,
    SSLTrainingConfig,
    build_random_init_probe_state,
    build_segment_sampler,
    list_ssl_checkpoints,
    load_precomputed_session_feature_stats_into_cache_context,
    plot_ssl_training_history,
    prepare_cache_context,
    recover_downstream_probe_state,
    recover_ssl_run_state_from_checkpoint,
    resolve_ssl_checkpoint_path,
    run_downstream_probe,
    run_probe_head_sweep,
    resume_ssl_training,
    run_ssl_training,
    display_probe_summaries,
    display_ssl_reconstruction_report,
    SWEEP_VITAL_COLUMNS,
    run_sigma_mask_probe_sweep,
)
from masked_ssl.probe import build_downstream_probe_problem

print("cwd:", Path.cwd())
print("repo dir exists:", REPO_DIR.exists(), REPO_DIR)
print("experiments dir exists:", EXPERIMENTS_DIR.exists(), EXPERIMENTS_DIR)
print("masked_ssl dir exists:", MASKED_SSL_DIR.exists(), MASKED_SSL_DIR)
print("ssl dir exists:", SSL_DIR.exists(), SSL_DIR)
print("SSL_AUTORESEARCH_CACHE_ROOT:", os.environ["SSL_AUTORESEARCH_CACHE_ROOT"])
print("SSL_AUTORESEARCH_OUTPUT_ROOT:", os.environ["SSL_AUTORESEARCH_OUTPUT_ROOT"])


Using existing repo: /content/utah-ssl
cwd: /content/utah-ssl
repo dir exists: True /content/utah-ssl
experiments dir exists: True /content/utah-ssl/analysis/active/ssl_experiments
masked_ssl dir exists: True /content/utah-ssl/analysis/active/ssl_experiments/masked_ssl
ssl dir exists: True /content/utah-ssl/analysis/active/transfer_benchmark/ssl_autoresearch
SSL_AUTORESEARCH_CACHE_ROOT: /content/drive/MyDrive/utah_ssl/data/cache_v1
SSL_AUTORESEARCH_OUTPUT_ROOT: /content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/masked_reconstruction


In [26]:
# Experiment config.

SEED = 7

SSL_STATE_MODE = "train"  # Set to "recover" to load a previous checkpoint.
GAUSSIAN_SMOOTHING_SIGMA_BINS = 2.0
SESSION_STATS_BIN_STRIDE = 2  # use every Nth bin when recomputing session stats
SESSION_STATS_VARIANT = "smoothed"  # one of {"smoothed", "unsmoothed", "none"}
STABLE_SSL_SESSION_STATS_PATH_UNSMOOTHED = Path("/content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/contrastive/precomputed_ssl_session_stats/session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_stable.pt")

def _sigma_tag_for_filename(value: float) -> str:
    value = float(value)
    text = f"{value:.6f}".rstrip("0").rstrip(".")
    if "." not in text:
        text = f"{text}.0"
    return text.replace("-", "m").replace(".", "p")

SMOOTHED_SIGMA_TAG = _sigma_tag_for_filename(GAUSSIAN_SMOOTHING_SIGMA_BINS)
STABLE_SSL_SESSION_STATS_PATH_SMOOTHED = Path(
    "/content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/contrastive/precomputed_ssl_session_stats/"
    f"session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_smooth_sigma{SMOOTHED_SIGMA_TAG}_stable.pt"
)

if SESSION_STATS_VARIANT == "smoothed":
    STABLE_SSL_SESSION_STATS_PATH = STABLE_SSL_SESSION_STATS_PATH_SMOOTHED
elif SESSION_STATS_VARIANT == "unsmoothed":
    STABLE_SSL_SESSION_STATS_PATH = STABLE_SSL_SESSION_STATS_PATH_UNSMOOTHED
elif SESSION_STATS_VARIANT == "none":
    STABLE_SSL_SESSION_STATS_PATH = None
else:
    raise ValueError("SESSION_STATS_VARIANT must be one of {'smoothed', 'unsmoothed', 'none'}")

if STABLE_SSL_SESSION_STATS_PATH is not None and not STABLE_SSL_SESSION_STATS_PATH.exists():
    print(
        "warning: selected STABLE_SSL_SESSION_STATS_PATH does not exist; "
        "falling back to recomputing session stats during cache preparation:",
        STABLE_SSL_SESSION_STATS_PATH,
    )
    STABLE_SSL_SESSION_STATS_PATH = None

SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH = None
SSL_RECOVERY_RUN_DIR = None
ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH = None

FEATURE_MODE = "tx_only"
SEGMENT_BINS = 80
PATCH_SIZE = 4
PATCH_STRIDE = 2
HIDDEN_SIZE = 256
S5_STATE_SIZE = 128
NUM_LAYERS = 4
DROPOUT = 0
BATCH_SIZE = 32
NUM_STEPS = 1500
LEARNING_RATE = 1e-4
WEIGHT_DECAY = 0
VAL_EVERY = 50
VAL_BATCHES = 10
CHECKPOINT_EVERY_STEPS = 500
DATASET_WEIGHT_ALPHA = 0.25
EXAMPLES_PER_SHARD = 8
POST_PROJ_NORM = "rms"
RECONSTRUCTION_HEAD_TYPE = "mlp"
BACKBONE_DIRECTION = "bidirectional"

MASK_UNIT = "patch"  # Set to "bin" for raw-bin contiguous masking.
MASK_TOKEN_PLACEMENT = "before_projection"
MASK_RATIO = 0.4
PATCH_SPAN_LENGTH_MIN = 2
PATCH_SPAN_LENGTH_MAX = 4
BIN_SPAN_LENGTH_MIN = PATCH_SPAN_LENGTH_MIN * PATCH_SIZE
BIN_SPAN_LENGTH_MAX = PATCH_SPAN_LENGTH_MAX * PATCH_SIZE
SPAN_LENGTH_MIN = PATCH_SPAN_LENGTH_MIN if MASK_UNIT == "patch" else BIN_SPAN_LENGTH_MIN
SPAN_LENGTH_MAX = PATCH_SPAN_LENGTH_MAX if MASK_UNIT == "patch" else BIN_SPAN_LENGTH_MAX
NUM_SPANS_MODE = "multiple"
ALLOW_BIN_FRACTIONAL_OVERLAP = True
RECONSTRUCT_TARGET = "raw_patch"
LOSS_MODE = "masked_only"

CACHE_ACCESS_MODE = "drive_direct"  # or "copy_to_local"
LOCAL_CACHE_BASE = "/content/utah_ssl_cache"
FORCE_RECOPY_LOCAL_CACHE = False
EXCLUDED_DATASETS = {"brain2text25"}
LOG_EVERY = 10

NORMALIZE_IMPL_VERSION = "session_featurewise_v1"
CACHE_PREPARE_NORMALIZE_IMPL_VERSION = (
    "segment_prefix_v1"
    if (NORMALIZE_IMPL_VERSION == "session_featurewise_v1" and STABLE_SSL_SESSION_STATS_PATH is not None)
    else NORMALIZE_IMPL_VERSION
)
NORMALIZE_CONTEXT_BINS = min(16, SEGMENT_BINS)
if NORMALIZE_IMPL_VERSION == "session_featurewise_v1" and GAUSSIAN_SMOOTHING_SIGMA_BINS > 0 and SESSION_STATS_VARIANT == "unsmoothed":
    print(
        "warning: using smoothed inputs with unsmoothed session stats; "
        "set SESSION_STATS_VARIANT='smoothed' for matched preprocessing."
    )

CACHE_ACCESS_CONFIG = CacheAccessConfig(
    mode=CACHE_ACCESS_MODE,
    local_cache_base=LOCAL_CACHE_BASE,
    force_recopy_local_cache=FORCE_RECOPY_LOCAL_CACHE,
    excluded_datasets=tuple(sorted(EXCLUDED_DATASETS)),
    seed=SEED,
    segment_bins=SEGMENT_BINS,
    normalize_context_bins=NORMALIZE_CONTEXT_BINS,
    normalize_impl_version=CACHE_PREPARE_NORMALIZE_IMPL_VERSION,
    gaussian_smoothing_sigma_bins=GAUSSIAN_SMOOTHING_SIGMA_BINS,
    session_stats_bin_stride=SESSION_STATS_BIN_STRIDE,
    examples_per_shard=EXAMPLES_PER_SHARD,
    feature_mode=FEATURE_MODE,
    boundary_key_mode="subject_if_available",
)

SSL_TRAINING_CONFIG = SSLTrainingConfig(
    seed=SEED,
    feature_mode=FEATURE_MODE,
    segment_bins=SEGMENT_BINS,
    patch_size=PATCH_SIZE,
    patch_stride=PATCH_STRIDE,
    hidden_size=HIDDEN_SIZE,
    s5_state_size=S5_STATE_SIZE,
    num_layers=NUM_LAYERS,
    dropout=DROPOUT,
    batch_size=BATCH_SIZE,
    num_steps=NUM_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=WEIGHT_DECAY,
    val_every=VAL_EVERY,
    val_batches=VAL_BATCHES,
    checkpoint_every_steps=CHECKPOINT_EVERY_STEPS,
    dataset_weight_alpha=DATASET_WEIGHT_ALPHA,
    examples_per_shard=EXAMPLES_PER_SHARD,
    log_every=LOG_EVERY,
    post_proj_norm=POST_PROJ_NORM,
    reconstruction_head_type=RECONSTRUCTION_HEAD_TYPE,
    backbone_direction=BACKBONE_DIRECTION,
    boundary_key_mode="subject_if_available",
    mask_unit=MASK_UNIT,
    mask_token_placement=MASK_TOKEN_PLACEMENT,
    mask_ratio=MASK_RATIO,
    span_length_min=SPAN_LENGTH_MIN,
    span_length_max=SPAN_LENGTH_MAX,
    num_spans_mode=NUM_SPANS_MODE,
    reconstruct_target=RECONSTRUCT_TARGET,
    loss_mode=LOSS_MODE,
    allow_bin_fractional_overlap=ALLOW_BIN_FRACTIONAL_OVERLAP,
)

print("Notebook workflow switches:", {
    "SSL_STATE_MODE": SSL_STATE_MODE,
    "SESSION_STATS_VARIANT": SESSION_STATS_VARIANT,
    "STABLE_SSL_SESSION_STATS_PATH": STABLE_SSL_SESSION_STATS_PATH,
    "STABLE_SSL_SESSION_STATS_PATH_UNSMOOTHED": STABLE_SSL_SESSION_STATS_PATH_UNSMOOTHED,
    "STABLE_SSL_SESSION_STATS_PATH_SMOOTHED": STABLE_SSL_SESSION_STATS_PATH_SMOOTHED,
    "SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH": SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
    "SSL_RECOVERY_RUN_DIR": SSL_RECOVERY_RUN_DIR,
    "ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH": ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH,
})
print("Masking config:", {
    "mask_unit": MASK_UNIT,
    "mask_token_placement": MASK_TOKEN_PLACEMENT,
    "mask_ratio": MASK_RATIO,
    "span_length_min": SPAN_LENGTH_MIN,
    "span_length_max": SPAN_LENGTH_MAX,
    "num_spans_mode": NUM_SPANS_MODE,
})
print("Patch config:", {
    "patch_size": PATCH_SIZE,
    "patch_stride": PATCH_STRIDE,
    "segment_bins": SEGMENT_BINS,
    "reconstruction_head_type": RECONSTRUCTION_HEAD_TYPE,
    "backbone_direction": BACKBONE_DIRECTION,
})
print("Feature config:", {
    "feature_mode": FEATURE_MODE,
    "normalize_impl_version": NORMALIZE_IMPL_VERSION,
    "cache_prepare_normalize_impl_version": CACHE_PREPARE_NORMALIZE_IMPL_VERSION,
    "gaussian_smoothing_sigma_bins": GAUSSIAN_SMOOTHING_SIGMA_BINS,
    "session_stats_bin_stride": SESSION_STATS_BIN_STRIDE,
})
print("CACHE_ACCESS_CONFIG:", CACHE_ACCESS_CONFIG)
print("SSL_TRAINING_CONFIG:", SSL_TRAINING_CONFIG)




Notebook workflow switches: {'SSL_STATE_MODE': 'train', 'SESSION_STATS_VARIANT': 'smoothed', 'STABLE_SSL_SESSION_STATS_PATH': PosixPath('/content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/contrastive/precomputed_ssl_session_stats/session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_smooth_sigma2p0_stable.pt'), 'STABLE_SSL_SESSION_STATS_PATH_UNSMOOTHED': PosixPath('/content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/contrastive/precomputed_ssl_session_stats/session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_stable.pt'), 'STABLE_SSL_SESSION_STATS_PATH_SMOOTHED': PosixPath('/content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/contrastive/precomputed_ssl_session_stats/session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_smooth_sigma2p0_stable.pt'), 'SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH': None, 'SSL_RECOVERY_RUN_DIR': None, 'ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH': None}
Masking config: {'mask_unit'

In [27]:
# Resolve cache access mode, summarize datasets, and build the reusable cache context.

import torch

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")

CACHE_CONTEXT = prepare_cache_context(
    cache_candidates=cache_candidates,
    config=CACHE_ACCESS_CONFIG,
)
FULL_DIM = CACHE_CONTEXT.full_dim

print("DEVICE:", DEVICE)
print("CACHE_CONTEXT cache_root:", CACHE_CONTEXT.cache_root)
print("FULL_DIM:", FULL_DIM)
print("session split summary:")
for dataset, summary in CACHE_CONTEXT.session_split_summary.items():
    print(
        f" - {dataset}: sessions={summary['total_sessions']} train_sessions={summary['train_sessions']} "
        f"val_sessions={summary['val_sessions']} train_examples={summary['train_examples']} "
        f"val_examples={summary['val_examples']}"
    )


using Drive-backed cache directly; skipping local copy
source: /content/drive/MyDrive/utah_ssl/data/cache_v1
source signature: 4331fe6a01f3
DEVICE: cuda
CACHE_CONTEXT cache_root: /content/drive/MyDrive/utah_ssl/data/cache_v1
FULL_DIM: 256
session split summary:
 - 000950: sessions=47 train_sessions=37 val_sessions=10 train_examples=569 val_examples=159
 - brain2text24: sessions=28 train_sessions=22 val_sessions=6 train_examples=12811 val_examples=3277
 - motor_data: sessions=21 train_sessions=16 val_sessions=5 train_examples=13611 val_examples=2809
 - plug_n_play: sessions=31 train_sessions=24 val_sessions=7 train_examples=1078 val_examples=237
 - unsupervised_cursor_recalibration_offline: sessions=103 train_sessions=82 val_sessions=21 train_examples=57000 val_examples=12157
 - unsupervised_cursor_recalibration_online: sessions=11 train_sessions=8 val_sessions=3 train_examples=72 val_examples=25
 - willett_handwriting: sessions=10 train_sessions=8 val_sessions=2 train_examples=4553 val

In [28]:
# Load precomputed SSL session-level featurewise z-scoring stats when available.

if STABLE_SSL_SESSION_STATS_PATH is not None:
    SSL_SESSION_STATS_STATE = load_precomputed_session_feature_stats_into_cache_context(
        cache_context=CACHE_CONTEXT,
        stats_path=STABLE_SSL_SESSION_STATS_PATH,
        normalize_impl_version=NORMALIZE_IMPL_VERSION,
    )
    print("Loaded cached SSL session stats from stable path.")
else:
    SSL_SESSION_STATS_STATE = {
        "stats_path": None,
        "metadata": {
            "source": "prepare_cache_context",
            "gaussian_smoothing_sigma_bins": float(GAUSSIAN_SMOOTHING_SIGMA_BINS),
            "session_stats_bin_stride": int(SESSION_STATS_BIN_STRIDE),
        },
        "session_count": len(CACHE_CONTEXT.session_feature_stats),
        "normalize_impl_version": CACHE_CONTEXT.normalize_impl_version,
    }
    print("Using session stats computed during cache preparation.")

print("session_count:", SSL_SESSION_STATS_STATE["session_count"])
print("normalize_impl_version:", SSL_SESSION_STATS_STATE["normalize_impl_version"])
print("metadata:", SSL_SESSION_STATS_STATE["metadata"])

print("expected_gaussian_smoothing_sigma_bins:", GAUSSIAN_SMOOTHING_SIGMA_BINS)
meta_sigma = SSL_SESSION_STATS_STATE["metadata"].get("gaussian_smoothing_sigma_bins")
print("stats_metadata_gaussian_smoothing_sigma_bins:", meta_sigma)
if meta_sigma is not None and abs(float(meta_sigma) - float(GAUSSIAN_SMOOTHING_SIGMA_BINS)) > 1e-6:
    print(
        "warning: loaded session stats metadata smoothing sigma does not match current config. "
        "This can skew normalization scale."
    )

print("expected_session_stats_bin_stride:", SESSION_STATS_BIN_STRIDE)
meta_stride = SSL_SESSION_STATS_STATE["metadata"].get("session_stats_bin_stride")
print("stats_metadata_session_stats_bin_stride:", meta_stride)
meta_stride_norm = None
if meta_stride is not None:
    try:
        meta_stride_norm = int(round(float(meta_stride)))
    except (TypeError, ValueError):
        print(
            "warning: loaded session stats metadata stride is not parseable as a number. "
            "This can skew normalization scale."
        )
if meta_stride_norm is not None and int(meta_stride_norm) != int(SESSION_STATS_BIN_STRIDE):
    print(
        "warning: loaded session stats metadata stride does not match current config. "
        "This can skew normalization scale."
    )





Loaded cached SSL session stats from stable path.
session_count: 251
normalize_impl_version: session_featurewise_v1
metadata: {'source': 's5_maskedreconstruction_in_memory', 'created_unix': 1776348862.3535125, 'normalize_impl_version': 'session_featurewise_v1', 'feature_mode': 'tx_only', 'gaussian_smoothing_sigma_bins': 2.0, 'session_stats_bin_stride': 1}
expected_gaussian_smoothing_sigma_bins: 2.0
stats_metadata_gaussian_smoothing_sigma_bins: 2.0
expected_session_stats_bin_stride: 2
stats_metadata_session_stats_bin_stride: 1


In [29]:
# # Quick batch inspection.

# INSPECT_SAMPLER = build_segment_sampler(
#     CACHE_CONTEXT,
#     "train",
#     batch_size=4,
#     seed=SEED,
#     segment_bins=SEGMENT_BINS,
#     dataset_weight_alpha=DATASET_WEIGHT_ALPHA,
#     examples_per_shard=EXAMPLES_PER_SHARD,
# )
# inspect_batch = INSPECT_SAMPLER.sample_batch()
# print("inspect_batch[x].shape:", tuple(inspect_batch["x"].shape))
# print("inspect_batch[lengths]:", inspect_batch["lengths"].tolist())
# print("inspect_batch[datasets]:", inspect_batch["datasets"])
# print("inspect_batch[sessions]:", inspect_batch["sessions"])


In [30]:
# TEMP: restrict SSL sampling to a single dataset for overfit debugging.
TARGET_DATASET = "brain2text24"

if TARGET_DATASET not in CACHE_CONTEXT.pretrain_datasets:
    raise ValueError(
        f"{TARGET_DATASET!r} is not in pretrain_datasets: {CACHE_CONTEXT.pretrain_datasets}"
    )

# Optional backup so you can restore later in-session.
if "_ORIG_PRETRAIN_DATASETS" not in globals():
    _ORIG_PRETRAIN_DATASETS = list(CACHE_CONTEXT.pretrain_datasets)

CACHE_CONTEXT.pretrain_datasets = [TARGET_DATASET]

# Clear cached sampling plans so samplers rebuild with the new dataset restriction.
CACHE_CONTEXT.sampling_plan_cache.clear()

train_n = len(CACHE_CONTEXT.split_rows_by_dataset["train"].get(TARGET_DATASET, []))
val_n = len(CACHE_CONTEXT.split_rows_by_dataset["val"].get(TARGET_DATASET, []))
print("Single-dataset sampler active:", CACHE_CONTEXT.pretrain_datasets)
print(f"{TARGET_DATASET} rows -> train={train_n}, val={val_n}")


Single-dataset sampler active: ['brain2text24']
brain2text24 rows -> train=12811, val=3277


In [31]:
# TEMP: restrict SSL sampling to one dataset + explicit train/val sessions.
import copy

TARGET_DATASET = "brain2text24"
TRAIN_SESSION_ID = "t12.2022.04.21"
VAL_SESSION_ID = "t12.2022.07.27"

if TRAIN_SESSION_ID == VAL_SESSION_ID:
    raise ValueError("TRAIN_SESSION_ID and VAL_SESSION_ID must be different.")

if TARGET_DATASET not in CACHE_CONTEXT.pretrain_datasets:
    raise ValueError(
        f"{TARGET_DATASET!r} is not in pretrain_datasets: {CACHE_CONTEXT.pretrain_datasets}"
    )

# One-time backup so you can restore later in this runtime.
if "_ORIG_CACHE_SPLITS" not in globals():
    _ORIG_CACHE_SPLITS = copy.deepcopy(CACHE_CONTEXT.split_rows_by_dataset)
    _ORIG_ROWS_BY_DATASET = copy.deepcopy(CACHE_CONTEXT.rows_by_dataset)
    _ORIG_PRETRAIN_DATASETS = list(CACHE_CONTEXT.pretrain_datasets)
    _ORIG_HAS_VAL_DATASETS = bool(CACHE_CONTEXT.has_val_datasets)
    _ORIG_SESSION_SPLIT_SUMMARY = copy.deepcopy(CACHE_CONTEXT.session_split_summary)

all_rows = list(_ORIG_ROWS_BY_DATASET[TARGET_DATASET])
train_rows = [row for row in all_rows if row.session_id == TRAIN_SESSION_ID]
val_rows = [row for row in all_rows if row.session_id == VAL_SESSION_ID]

if not train_rows:
    raise RuntimeError(
        f"No rows found for train session {TRAIN_SESSION_ID!r} in {TARGET_DATASET!r}."
    )
if not val_rows:
    raise RuntimeError(
        f"No rows found for val session {VAL_SESSION_ID!r} in {TARGET_DATASET!r}."
    )

# Restrict active dataset set.
CACHE_CONTEXT.pretrain_datasets = [TARGET_DATASET]

# Force explicit train/val session split for this toy experiment.
CACHE_CONTEXT.split_rows_by_dataset["train"][TARGET_DATASET] = train_rows
CACHE_CONTEXT.split_rows_by_dataset["val"][TARGET_DATASET] = val_rows

# Keep only these two sessions in rows_by_dataset so model/session-key discovery is aligned.
CACHE_CONTEXT.rows_by_dataset[TARGET_DATASET] = [
    row for row in all_rows if row.session_id in {TRAIN_SESSION_ID, VAL_SESSION_ID}
]

CACHE_CONTEXT.session_split_summary[TARGET_DATASET] = {
    "total_sessions": 2,
    "train_sessions": 1,
    "val_sessions": 1,
    "val_eligible": True,
    "train_examples": len(train_rows),
    "val_examples": len(val_rows),
    "train_session_ids": [TRAIN_SESSION_ID],
    "val_session_ids": [VAL_SESSION_ID],
}

CACHE_CONTEXT.has_val_datasets = True

# Clear cached sampling plans so train/val samplers rebuild with this forced split.
CACHE_CONTEXT.sampling_plan_cache.clear()

print("Two-session toy split active")
print("dataset:", TARGET_DATASET)
print("train session:", TRAIN_SESSION_ID, "rows:", len(train_rows))
print("val session:", VAL_SESSION_ID, "rows:", len(val_rows))
print("has_val_datasets:", CACHE_CONTEXT.has_val_datasets)


Two-session toy split active
dataset: brain2text24
train session: t12.2022.04.21 rows: 1320
val session: t12.2022.07.27 rows: 682
has_val_datasets: True


In [32]:
# Acquire SSL state: either train now or recover from a saved checkpoint.

if NORMALIZE_IMPL_VERSION == "session_featurewise_v1" and not CACHE_CONTEXT.session_feature_stats:
    raise RuntimeError("Run the stable session-stats cell first so session-level z-scoring is loaded into CACHE_CONTEXT.")

if SSL_STATE_MODE == "train":
    SSL_RUN_STATE = run_ssl_training(
        cache_context=CACHE_CONTEXT,
        config=SSL_TRAINING_CONFIG,
        output_root=OUTPUT_ROOT,
        device=DEVICE,
    )
elif SSL_STATE_MODE == "recover":
    resolved_checkpoint_path = resolve_ssl_checkpoint_path(
        output_root=OUTPUT_ROOT,
        explicit_checkpoint_path=SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
        run_dir=SSL_RECOVERY_RUN_DIR,
    )
    SSL_RUN_STATE = recover_ssl_run_state_from_checkpoint(
        checkpoint_path=resolved_checkpoint_path,
        cache_context=CACHE_CONTEXT,
        device=DEVICE,
        fallback_config=SSL_TRAINING_CONFIG,
    )
else:
    raise ValueError(f"Unsupported SSL_STATE_MODE: {SSL_STATE_MODE}")

print("run_name:", SSL_RUN_STATE["run_name"])
print("run_dir:", SSL_RUN_STATE["run_dir"])
print("checkpoint_path:", SSL_RUN_STATE["checkpoint_path"])
print("best_score:", SSL_RUN_STATE["best_score"])
print("best_step:", SSL_RUN_STATE["best_step"])


step=001 train_loss=0.9549 mask_ratio=0.410 masked_token_ratio=0.410 grad_norm=78.0223 sample_s=0.07 model_s=0.41
step=010 train_loss=0.8735 mask_ratio=0.410 masked_token_ratio=0.410 grad_norm=1.0873 sample_s=0.03 model_s=0.40
step=020 train_loss=0.9426 mask_ratio=0.410 masked_token_ratio=0.410 grad_norm=0.9105 sample_s=0.03 model_s=0.41
step=030 train_loss=0.9575 mask_ratio=0.410 masked_token_ratio=0.410 grad_norm=0.8839 sample_s=0.04 model_s=0.42
step=040 train_loss=0.9106 mask_ratio=0.410 masked_token_ratio=0.410 grad_norm=1.3987 sample_s=0.04 model_s=0.41
step=050 train_loss=0.9281 mask_ratio=0.410 masked_token_ratio=0.410 grad_norm=1.1801 sample_s=0.03 model_s=0.41
step=050 val_loss=0.9909 val_mask_ratio=0.410 val_masked_token_ratio=0.410
step=060 train_loss=0.8994 mask_ratio=0.410 masked_token_ratio=0.410 grad_norm=0.8775 sample_s=0.03 model_s=0.41
step=070 train_loss=0.8941 mask_ratio=0.410 masked_token_ratio=0.410 grad_norm=1.2255 sample_s=0.03 model_s=0.41
step=080 train_loss=

In [16]:
# SSL RESUME cell: continue training from the current in-memory SSL_RUN_STATE.

RESUME_ADDITIONAL_STEPS = 1000

if "SSL_RUN_STATE" not in globals():
    if ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH is not None:
        resolved_checkpoint_path = Path(ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH)
        if not resolved_checkpoint_path.exists():
            raise FileNotFoundError(
                f"ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH does not exist: {resolved_checkpoint_path}"
            )
    else:
        resolved_checkpoint_path = resolve_ssl_checkpoint_path(
            output_root=OUTPUT_ROOT,
            explicit_checkpoint_path=SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
            run_dir=SSL_RECOVERY_RUN_DIR,
        )
    SSL_RUN_STATE = recover_ssl_run_state_from_checkpoint(
        checkpoint_path=resolved_checkpoint_path,
        cache_context=CACHE_CONTEXT,
        device=DEVICE,
        fallback_config=SSL_TRAINING_CONFIG,
    )
    print("Recovered SSL_RUN_STATE from checkpoint:", resolved_checkpoint_path)

SSL_RUN_STATE = resume_ssl_training(
    run_state=SSL_RUN_STATE,
    additional_steps=RESUME_ADDITIONAL_STEPS,
    cache_context=CACHE_CONTEXT,
    device=DEVICE,
)


Continuing in-memory masked SSL training
 - run_dir: /content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/masked_reconstruction/colab_s5_masked_reconstruction_seg80_20260416T152031Z
 - start_step: 4000
 - target_step: 5000
 - mask_ratio: 0.15
step=4001 train_loss=0.4887 mask_ratio=0.154 masked_token_ratio=0.154 grad_norm=1.1584 sample_s=0.04 model_s=0.41
step=4010 train_loss=0.4926 mask_ratio=0.154 masked_token_ratio=0.154 grad_norm=1.1358 sample_s=0.03 model_s=0.40
step=4020 train_loss=0.4823 mask_ratio=0.154 masked_token_ratio=0.154 grad_norm=1.1450 sample_s=0.04 model_s=0.40
step=4030 train_loss=0.5227 mask_ratio=0.154 masked_token_ratio=0.154 grad_norm=1.6602 sample_s=0.03 model_s=0.40
step=4040 train_loss=0.5019 mask_ratio=0.154 masked_token_ratio=0.154 grad_norm=1.1851 sample_s=0.03 model_s=0.39
step=4050 train_loss=0.5199 mask_ratio=0.154 masked_token_ratio=0.154 grad_norm=1.2265 sample_s=0.04 model_s=0.41
step=4050 val_loss=0.7295 val_mask_ratio=0.154 val_masked_token_ratio=

In [37]:
# Resolve the active checkpoint path used by downstream probe recovery.

if "SSL_RUN_STATE" not in globals() and ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH is None:
    resolved_checkpoint_path = resolve_ssl_checkpoint_path(
        output_root=OUTPUT_ROOT,
        explicit_checkpoint_path=SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
        run_dir=SSL_RECOVERY_RUN_DIR,
    )
    SSL_RUN_STATE = recover_ssl_run_state_from_checkpoint(
        checkpoint_path=resolved_checkpoint_path,
        cache_context=CACHE_CONTEXT,
        device=DEVICE,
        fallback_config=SSL_TRAINING_CONFIG,
    )
    print("Recovered SSL_RUN_STATE from checkpoint:", resolved_checkpoint_path)

if ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH is None:
    ACTIVE_SSL_CHECKPOINT_PATH = Path(SSL_RUN_STATE["checkpoint_path"])
else:
    ACTIVE_SSL_CHECKPOINT_PATH = Path(ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH)
    if not ACTIVE_SSL_CHECKPOINT_PATH.exists():
        raise FileNotFoundError(
            f"ACTIVE_SSL_CHECKPOINT_OVERRIDE_PATH does not exist: {ACTIVE_SSL_CHECKPOINT_PATH}"
        )

ACTIVE_SSL_CHECKPOINT_RUN_DIR = (
    ACTIVE_SSL_CHECKPOINT_PATH.parent.parent
    if ACTIVE_SSL_CHECKPOINT_PATH.parent.name == "checkpoints"
    else ACTIVE_SSL_CHECKPOINT_PATH.parent
)

print("ACTIVE_SSL_CHECKPOINT_PATH:", ACTIVE_SSL_CHECKPOINT_PATH)
print("ACTIVE_SSL_CHECKPOINT_RUN_DIR:", ACTIVE_SSL_CHECKPOINT_RUN_DIR)



ACTIVE_SSL_CHECKPOINT_PATH: /content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/masked_reconstruction/colab_s5_masked_reconstruction_seg80_20260418T144825Z/checkpoint_final.pt
ACTIVE_SSL_CHECKPOINT_RUN_DIR: /content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/masked_reconstruction/colab_s5_masked_reconstruction_seg80_20260418T144825Z


In [ ]:
# Plot train / val curves and print the compact SSL reconstruction scorecard.

if "SSL_RUN_STATE" not in globals():
    resolved_checkpoint_path = resolve_ssl_checkpoint_path(
        output_root=OUTPUT_ROOT,
        explicit_checkpoint_path=SSL_RECOVERY_EXPLICIT_CHECKPOINT_PATH,
        run_dir=SSL_RECOVERY_RUN_DIR,
    )
    SSL_RUN_STATE = recover_ssl_run_state_from_checkpoint(
        checkpoint_path=resolved_checkpoint_path,
        cache_context=CACHE_CONTEXT,
        device=DEVICE,
        fallback_config=SSL_TRAINING_CONFIG,
    )
    print("Recovered SSL_RUN_STATE from checkpoint:", resolved_checkpoint_path)

SSL_RECON_SCORECARD = display_ssl_reconstruction_report(
    SSL_RUN_STATE,
    zero_baseline_df=globals().get("ZERO_BASELINE_DF"),
)


In [ ]:
# Optional: list saved SSL checkpoints for the active run.


CHECKPOINT_LIST_RUN_DIR = Path(ACTIVE_SSL_CHECKPOINT_RUN_DIR)
AVAILABLE_SSL_CHECKPOINTS = list_ssl_checkpoints(CHECKPOINT_LIST_RUN_DIR)
display(pd.DataFrame(AVAILABLE_SSL_CHECKPOINTS))


## Downstream Probe Workflow

These cells keep the same held-out `Brain2Text25` workflow as the other `s5` notebooks.
The main apples-to-apples frozen comparison is:

- pretrained masked-reconstruction encoder, frozen, linear probe
- random-init encoder, frozen, same linear probe


In [ ]:
# Configure and preview the held-out Brain2Text25 probe split.

PROBE_SESSION_LIMIT = 8
PROBE_TARGET_SESSION_COUNT = 4
PROBE_BATCH_SIZE = 8
PROBE_BUDGET_SECONDS = 240
PROBE_MAX_STEPS = 100
PROBE_HEAD_LEARNING_RATE = 1e-3
ENCODER_LEARNING_RATE = 3e-4
PROBE_WEIGHT_DECAY = 1e-2

DOWNSTREAM_PROBE_CONFIG = DownstreamProbeConfig(
    enabled=True,
    seed=SEED,
    session_limit=PROBE_SESSION_LIMIT,
    target_session_count=PROBE_TARGET_SESSION_COUNT,
    probe_batch_size=PROBE_BATCH_SIZE,
    probe_budget_seconds=PROBE_BUDGET_SECONDS,
    max_probe_steps=PROBE_MAX_STEPS,
    probe_head_learning_rate=PROBE_HEAD_LEARNING_RATE,
    encoder_learning_rate=None,
    weight_decay=PROBE_WEIGHT_DECAY,
    probe_head_type="linear",
)

downstream_problem_preview = build_downstream_probe_problem(
    cache_root=Path(CACHE_CONTEXT.cache_root),
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    feature_mode=FEATURE_MODE,
)
print("eligible_sessions:", len(downstream_problem_preview["eligible_entries"]))
print("preview_source_sessions:", [entry.session_base for entry in downstream_problem_preview["split"].train])
print("preview_target_sessions:", [entry.session_base for entry in downstream_problem_preview["split"].val])


eligible_sessions: 45
preview_source_sessions: ['t15.2024.07.21', 't15.2024.07.28', 't15.2025.01.10', 't15.2025.01.12']
preview_target_sessions: ['t15.2025.03.14', 't15.2025.03.16', 't15.2025.03.30', 't15.2025.04.13']


In [ ]:
# Shared downstream experiment helpers.

FROZEN_LINEAR_PROBE_OVERRIDES = {"probe_head_type": "linear"}
QUICK_CONV1D_PROBE_OVERRIDES = {
    "probe_head_type": "conv1d",
    "probe_conv_hidden_size": HIDDEN_SIZE,
    "probe_conv_kernel_size": 3,
    "probe_budget_seconds": 120,
    "max_probe_steps": 50,
}
B2_LINEAR_PROBE_OVERRIDES = {
    "probe_head_type": "linear",
    "probe_head_learning_rate": PROBE_HEAD_LEARNING_RATE,
    "encoder_learning_rate": ENCODER_LEARNING_RATE,
    "weight_decay": PROBE_WEIGHT_DECAY,
    "probe_budget_seconds": 600,
    "max_probe_steps": 800,
}

def build_notebook_random_probe_state(*, seed_offset: int = 0):
    return build_random_init_probe_state(
        reference_config=DOWNSTREAM_PROBE_DEFAULT_STATE["checkpoint_config"],
        input_dim=DOWNSTREAM_PROBE_DEFAULT_STATE["input_dim"],
        seed=SEED + int(seed_offset),
        base_run_dir=DOWNSTREAM_PROBE_BASE_RUN_DIR,
    )


In [38]:
# Recover the default SSL encoder and prepare reusable downstream probe state.

DOWNSTREAM_PROBE_DEFAULT_STATE = recover_downstream_probe_state(
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    output_root=OUTPUT_ROOT,
    input_dim=FULL_DIM,
    default_checkpoint_config=SSL_TRAINING_CONFIG.checkpoint_config(),
    in_memory_model=SSL_RUN_STATE["model"] if "SSL_RUN_STATE" in globals() else None,
    current_checkpoint_path=Path(ACTIVE_SSL_CHECKPOINT_PATH),
    current_run_dir=Path(ACTIVE_SSL_CHECKPOINT_RUN_DIR),
)

DOWNSTREAM_PROBE_BASE_RUN_DIR = Path(DOWNSTREAM_PROBE_DEFAULT_STATE["base_run_dir"])
print("DOWNSTREAM_PROBE_ENCODER_SOURCE:", DOWNSTREAM_PROBE_DEFAULT_STATE["source"])
print("DOWNSTREAM_PROBE_BASE_RUN_DIR:", DOWNSTREAM_PROBE_BASE_RUN_DIR)
print("DOWNSTREAM_PROBE_CHECKPOINT_PATH:", DOWNSTREAM_PROBE_DEFAULT_STATE["checkpoint_path"])


DOWNSTREAM_PROBE_ENCODER_SOURCE: checkpoint
DOWNSTREAM_PROBE_BASE_RUN_DIR: /content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/masked_reconstruction/colab_s5_masked_reconstruction_seg80_20260418T144825Z
DOWNSTREAM_PROBE_CHECKPOINT_PATH: /content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/masked_reconstruction/colab_s5_masked_reconstruction_seg80_20260418T144825Z/checkpoint_final.pt


In [ ]:
# Frozen SSL checkpoint linear probe.

SSL_CHECKPOINT_LINEAR_SUMMARY = run_downstream_probe(
    probe_state=DOWNSTREAM_PROBE_DEFAULT_STATE,
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    cache_root=Path(CACHE_CONTEXT.cache_root),
    device=DEVICE,
    variant_prefix="ssl_checkpoint",
    artifact_prefix="ssl_checkpoint",
    train_encoder=False,
    probe_overrides=FROZEN_LINEAR_PROBE_OVERRIDES,
)
display_probe_summaries(SSL_CHECKPOINT_LINEAR_SUMMARY)


In [ ]:
# Random-init frozen linear probe baseline.

RANDOM_INIT_FROZEN_STATE = build_notebook_random_probe_state(seed_offset=1000)
RANDOM_INIT_LINEAR_SUMMARY = run_downstream_probe(
    probe_state=RANDOM_INIT_FROZEN_STATE,
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    cache_root=Path(CACHE_CONTEXT.cache_root),
    device=DEVICE,
    variant_prefix="random_init",
    artifact_prefix="random_init",
    train_encoder=False,
    probe_overrides=FROZEN_LINEAR_PROBE_OVERRIDES,
)
display_probe_summaries(SSL_CHECKPOINT_LINEAR_SUMMARY, RANDOM_INIT_LINEAR_SUMMARY)


In [ ]:
# Optional stronger frozen-head diagnostic.

SSL_CHECKPOINT_CONV1D_SUMMARY = run_downstream_probe(
    probe_state=DOWNSTREAM_PROBE_DEFAULT_STATE,
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    cache_root=Path(CACHE_CONTEXT.cache_root),
    device=DEVICE,
    variant_prefix="ssl_checkpoint",
    artifact_prefix="ssl_checkpoint",
    train_encoder=False,
    probe_overrides=QUICK_CONV1D_PROBE_OVERRIDES,
)
display_probe_summaries(SSL_CHECKPOINT_CONV1D_SUMMARY)


In [ ]:
# Full fine-tuning from the masked SSL checkpoint.

SSL_CHECKPOINT_B2_SUMMARY = run_downstream_probe(
    probe_state=DOWNSTREAM_PROBE_DEFAULT_STATE,
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    cache_root=Path(CACHE_CONTEXT.cache_root),
    device=DEVICE,
    variant_prefix="ssl_checkpoint_b2",
    artifact_prefix="ssl_checkpoint_b2",
    train_encoder=True,
    probe_overrides=B2_LINEAR_PROBE_OVERRIDES,
)
display_probe_summaries(SSL_CHECKPOINT_B2_SUMMARY)


In [ ]:
# Full fine-tuning from random initialization.

RANDOM_INIT_B2_STATE = build_notebook_random_probe_state(seed_offset=2000)
RANDOM_INIT_B2_SUMMARY = run_downstream_probe(
    probe_state=RANDOM_INIT_B2_STATE,
    probe_config=DOWNSTREAM_PROBE_CONFIG,
    cache_root=Path(CACHE_CONTEXT.cache_root),
    device=DEVICE,
    variant_prefix="random_init_b2",
    artifact_prefix="random_init_b2",
    train_encoder=True,
    probe_overrides=B2_LINEAR_PROBE_OVERRIDES,
)
display_probe_summaries(SSL_CHECKPOINT_B2_SUMMARY, RANDOM_INIT_B2_SUMMARY)


In [ ]:
# Compact comparison table for the main summaries collected above.

MAIN_SUMMARIES = [
    globals().get("SSL_CHECKPOINT_LINEAR_SUMMARY"),
    globals().get("RANDOM_INIT_LINEAR_SUMMARY"),
    globals().get("SSL_CHECKPOINT_B2_SUMMARY"),
    globals().get("RANDOM_INIT_B2_SUMMARY"),
]
display_probe_summaries(*MAIN_SUMMARIES)


In [ ]:
# TEMP: evaluate one session directly from rows_by_dataset (ignores train/val split maps).
import random
import numpy as np

from masked_ssl.cache import sample_base_segment, stack_segment_batch
from masked_ssl.training import evaluate_model

TARGET_DATASET = "brain2text24"
TARGET_SESSION = "t12.2022.08.18"
EVAL_BATCHES = 40
EVAL_SEED = 123

cfg = dict(SSL_RUN_STATE["config"])

all_rows = CACHE_CONTEXT.rows_by_dataset.get(TARGET_DATASET, [])
session_rows = [r for r in all_rows if r.session_id == TARGET_SESSION]
if len(session_rows) == 0:
    raise RuntimeError(f"No rows found in rows_by_dataset for {TARGET_DATASET}:{TARGET_SESSION}")

segment_bins = int(cfg["segment_bins"])
valid_rows = [r for r in session_rows if int(r.n_time_bins) >= segment_bins]
if len(valid_rows) == 0:
    raise RuntimeError(f"No valid rows with n_time_bins >= {segment_bins} for {TARGET_SESSION}")

weights = np.array([max(1, int(r.n_time_bins) - segment_bins + 1) for r in valid_rows], dtype=np.float64)
weights /= weights.sum()

class SingleSessionSampler:
    def __init__(self, cache_context, rows, probs, *, batch_size, segment_bins, seed):
        self.cache_context = cache_context
        self.rows = rows
        self.probs = probs
        self.batch_size = int(batch_size)
        self.segment_bins = int(segment_bins)
        self.py_rng = random.Random(int(seed))
        self.np_rng = np.random.default_rng(int(seed))

    def sample_batch(self, batch_size=None):
        bs = self.batch_size if batch_size is None else int(batch_size)
        idx = self.np_rng.choice(len(self.rows), size=bs, replace=True, p=self.probs)
        samples = [
            sample_base_segment(
                self.cache_context,
                self.rows[int(i)],
                segment_bins=self.segment_bins,
                py_rng=self.py_rng,
            )
            for i in np.atleast_1d(idx)
        ]
        return stack_segment_batch(samples)

sampler = SingleSessionSampler(
    CACHE_CONTEXT,
    valid_rows,
    weights,
    batch_size=int(cfg["batch_size"]),
    segment_bins=segment_bins,
    seed=EVAL_SEED,
)

metrics = evaluate_model(
    SSL_RUN_STATE["model"],
    sampler,
    num_batches=int(EVAL_BATCHES),
    device=DEVICE,
    mask_unit=str(cfg["mask_unit"]),
    mask_token_placement=str(cfg["mask_token_placement"]),
    mask_ratio=float(cfg["mask_ratio"]),
    span_length_min=int(cfg["span_length_min"]),
    span_length_max=int(cfg["span_length_max"]),
    num_spans_mode=str(cfg["num_spans_mode"]),
    allow_bin_fractional_overlap=bool(cfg["allow_bin_fractional_overlap"]),
)

print(f"toy_eval dataset={TARGET_DATASET} session={TARGET_SESSION} (split-agnostic)")
for k in ["loss", "masked_token_full_patch_mse", "actual_mask_ratio", "masked_token_ratio", "masked_prediction_std", "masked_target_std"]:
    print(f"{k}: {metrics[k]:.6f}")


RuntimeError: No rows found in rows_by_dataset for brain2text24:t12.2022.08.18

## Notes

This notebook intentionally omits the contrastive retrieval diagnostics from the older `s5` notebooks.
For this experiment, the primary scorecard is held-out frozen phoneme transfer, with masked-reconstruction loss used as a training diagnostic.


In [22]:
# Cell 1: Sweep + brain2text24 single-session probe setup

# Sweep controls
SWEEP_SIGMAS = [1.5, 2.0, 2.5]
SWEEP_MASK_RATIOS = [0.25]
SWEEP_SSL_STEPS = 1200

# Probe controls
PROBE_DATASET = "brain2text24"
PROBE_SESSION_ID = None  # set explicitly (e.g. "t12.2022.07.27") or leave None to auto-pick
PROBE_MAX_STEPS = 100
PROBE_BATCH_SIZE = 8
PROBE_HEAD_LR = 1e-3
PROBE_WEIGHT_DECAY = 1e-2
PROBE_TRAIN_ENCODER = False  # frozen encoder probe
PROBE_ALLOW_NONE_AS_TRAIN = False

# Training split (same toy split style you were using)
TRAIN_DATASET = globals().get("TARGET_DATASET", "brain2text24")
TRAIN_SESSION_ID = globals().get("TRAIN_SESSION_ID", "t12.2022.04.21")
VAL_SESSION_ID = globals().get("VAL_SESSION_ID", "t12.2022.07.27")

SKIP_SIGMA_IF_STATS_MISSING = True  # True => skip sigma with no precomputed stats
SESSION_STATS_DIR = Path("/content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/contrastive/precomputed_ssl_session_stats")
REQUIRE_PRECOMPUTED_STATS_FOR_SWEEP = True


In [ ]:
# Cell 2: Run the configured sigma x mask_ratio sweep and print compact vital statistics.

from IPython.display import display

SWEEP_RESULTS_DF, SWEEP_CONTEXTS_BY_SIGMA = run_sigma_mask_probe_sweep(
    sweep_sigmas=SWEEP_SIGMAS,
    sweep_mask_ratios=SWEEP_MASK_RATIOS,
    sweep_ssl_steps=SWEEP_SSL_STEPS,
    cache_candidates=cache_candidates,
    base_cache_config=CACHE_ACCESS_CONFIG,
    base_training_config=SSL_TRAINING_CONFIG,
    output_root=OUTPUT_ROOT,
    device=DEVICE,
    active_cache_context=globals().get("CACHE_CONTEXT"),
    active_sigma=float(globals().get("GAUSSIAN_SMOOTHING_SIGMA_BINS", float("nan"))),
    loaded_session_stats_state=globals().get("SSL_SESSION_STATS_STATE"),
    stable_stats_path=globals().get("STABLE_SSL_SESSION_STATS_PATH"),
    session_stats_dir=SESSION_STATS_DIR,
    normalize_impl_version=NORMALIZE_IMPL_VERSION,
    train_dataset=TRAIN_DATASET,
    train_session_id=TRAIN_SESSION_ID,
    val_session_id=VAL_SESSION_ID,
    probe_dataset=PROBE_DATASET,
    probe_session_id=PROBE_SESSION_ID,
    probe_max_steps=PROBE_MAX_STEPS,
    probe_batch_size=PROBE_BATCH_SIZE,
    probe_head_lr=PROBE_HEAD_LR,
    probe_weight_decay=PROBE_WEIGHT_DECAY,
    probe_train_encoder=PROBE_TRAIN_ENCODER,
    probe_allow_none_as_train=PROBE_ALLOW_NONE_AS_TRAIN,
    skip_sigma_if_stats_missing=SKIP_SIGMA_IF_STATS_MISSING,
    require_precomputed_stats_for_sweep=REQUIRE_PRECOMPUTED_STATS_FOR_SWEEP,
    boundary_key_mode="subject_if_available",
)

print("\nSweep complete.")
display(SWEEP_RESULTS_DF[SWEEP_VITAL_COLUMNS])


In [24]:
# Compute + save precomputed session stats for sigma=1.0 (smoothed), with stride-aware filename.

import time
from dataclasses import replace

SIGMA_TO_BUILD = 1.0
FORCE_OVERWRITE = False

def _sigma_tag_for_filename(value: float) -> str:
    text = f"{float(value):.6f}".rstrip("0").rstrip(".")
    if "." not in text:
        text = f"{text}.0"
    return text.replace("-", "m").replace(".", "p")

# Use the same directory your sweep cell searches.
save_dir = Path(globals().get(
    "SESSION_STATS_DIR",
    "/content/drive/MyDrive/utah_ssl/outputs/ssl_experiments/contrastive/precomputed_ssl_session_stats",
))
save_dir.mkdir(parents=True, exist_ok=True)

stride = int(globals().get("SESSION_STATS_BIN_STRIDE", 1))
sigma_tag = _sigma_tag_for_filename(SIGMA_TO_BUILD)

# Match the sweep resolver's preferred filename pattern.
out_path = save_dir / (
    "session_feature_stats_session_featurewise_v1_refds000950_cap126682_tx256_sbp256_"
    f"smooth_sigma{sigma_tag}_stride{stride}_stable.pt"
)

if out_path.exists() and not FORCE_OVERWRITE:
    print("Already exists, not overwriting:", out_path)
else:
    # Build a fresh context that COMPUTES session_featurewise stats at sigma=1.0.
    build_cfg = replace(
        CACHE_ACCESS_CONFIG,
        normalize_impl_version="session_featurewise_v1",
        gaussian_smoothing_sigma_bins=float(SIGMA_TO_BUILD),
        session_stats_bin_stride=int(stride),
    )

    t0 = time.time()
    build_ctx = prepare_cache_context(
        cache_candidates=cache_candidates,
        config=build_cfg,
    )
    elapsed = time.time() - t0

    stats = build_ctx.session_feature_stats
    if not stats:
        raise RuntimeError("No session_feature_stats were computed.")

    payload = {
        "session_feature_stats": {
            str(k): (mean.detach().cpu(), std.detach().cpu())
            for k, (mean, std) in stats.items()
        },
        "metadata": {
            "source": "s5_maskedreconstruction_sigma_builder",
            "created_unix": time.time(),
            "normalize_impl_version": "session_featurewise_v1",
            "feature_mode": str(build_cfg.feature_mode),
            "gaussian_smoothing_sigma_bins": float(SIGMA_TO_BUILD),
            "session_stats_bin_stride": int(stride),
            "cache_root": str(build_ctx.cache_root),
            "session_count": int(len(stats)),
            "build_seconds": float(elapsed),
        },
    }

    torch.save(payload, out_path)
    print("Saved:", out_path)
    print("session_count:", len(stats))
    print("build_seconds:", round(elapsed, 1))

# Optional: point notebook at this file immediately.
print("Use this in sweep:", out_path)


using Drive-backed cache directly; skipping local copy
source: /content/drive/MyDrive/utah_ssl/data/cache_v1
source signature: 4331fe6a01f3
computing SSL session-level featurewise z-scoring stats...
 session_stats=1/251 current=000950:sub-T5-held-in-calib_ses-20220518
 session_stats=25/251 current=000950:sub-T5-held-in-minival_ses-20220601
 session_stats=50/251 current=brain2text24:t12.2022.04.28
 session_stats=75/251 current=brain2text24:t12.2022.08.25
 session_stats=100/251 current=plug_n_play:t5.2022.06.01__seed_model_training
 session_stats=125/251 current=plug_n_play:t5.2022.12.15__recalibration
 session_stats=150/251 current=unsupervised_cursor_recalibration_offline:t5.2017.02.22
 session_stats=175/251 current=unsupervised_cursor_recalibration_offline:t5.2018.07.06
 session_stats=200/251 current=unsupervised_cursor_recalibration_offline:t5.2019.04.08
 session_stats=225/251 current=unsupervised_cursor_recalibration_offline:t5.2021.07.07
 session_stats=250/251 current=willett_handw